# 01 - Generate prompts

Attribute-based, **reproducible** prompt generation for the 5 plate-state classes.
Produces `shlomi/data/prompts.json` (keyed by class). This step is pure text - no GPU needed.

The 5 classes: `clean`, `empty`, `finished_leftovers`, `full` (a moderate-to-full
serving), and `unclassified`. The first four are *content* classes with their own
attribute-based prompts. `unclassified` has **no prompts of its own** - it borrows
random prompts from the other classes, and in `03_degrade_and_augment` those images
are corrupted until the plate state is unreadable.

All attribute pools, the prompt template, the class list and the seed live in this
workspace's `utils.py` (the single source of truth). This notebook just drives them.

In [ ]:
import sys
from pathlib import Path

# Self-contained shlomi/ workspace: utils.py lives next to this notebook.
# Make it importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / 'shlomi'):
    if (_cand / 'utils.py').exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))
        break
import utils

print('Classes        :', utils.CLASS_NAMES)
print('Content classes:', utils.CONTENT_CLASSES)
print('Seed           :', utils.SEED)

## Build and save the prompts

`300` prompts per class - we have the compute budget, so we favor maximal prompt
diversity (well above the rubric's 60-80 minimum). Each prompt spawns one or more
images in step 02. `unclassified` borrows its 300 prompts from the content classes.

In [ ]:
N_PER_CLASS = 300  # we have the compute budget, so favor maximal prompt diversity

prompts = utils.build_prompts(n_per_class=N_PER_CLASS, seed=utils.SEED)
utils.save_prompts(prompts)  # writes shlomi/data/prompts.json

print(f'Saved {sum(len(v) for v in prompts.values())} prompts to {utils.PROMPTS_PATH}')
for cls in utils.CLASS_NAMES:
    print(f'  {cls:20s} {len(prompts[cls])} prompts  ->  {utils.to_binary(cls)}')

## Inspect 5 prompts per class (manual quality check)

Skim these before generating any images. Check that `clean` looks clearly *pristine*
and `empty` clearly *used* - that pair is the subtlest to keep separable. The
`unclassified` entries are deliberately **borrowed** from the other classes; they
only become unidentifiable after the heavy corruption in step 03.

In [ ]:
for cls in utils.CLASS_NAMES:
    print()
    print(f'=== {cls}  ({utils.to_binary(cls)}) ===')
    for p in prompts[cls][:5]:
        print(' -', p)

## Sanity checks

In [ ]:
assert set(prompts) == set(utils.CLASS_NAMES), 'missing or extra classes'
for cls, ps in prompts.items():
    assert len(ps) == N_PER_CLASS, f'{cls}: expected {N_PER_CLASS}, got {len(ps)}'
    assert len(set(ps)) == len(ps), f'{cls}: duplicate prompts'

# unclassified must be borrowed from the content classes
content_pool = set(p for c in utils.CONTENT_CLASSES for p in prompts[c])
assert all(p in content_pool for p in prompts[utils.UNCLASSIFIED]), 'unclassified not borrowed'
print('All sanity checks passed:', N_PER_CLASS, 'unique prompts x', len(utils.CLASS_NAMES), 'classes')